<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "430"
iso_country = "Liberia"
year = 2025
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 5.250*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Liberia': 

Number of trade records present in dataset: 15882
Number of partner countries present in dataset: 147
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1887.9m
Total weight within the dataset: 2018.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,430,Liberia,156,China,Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,2067.83487,NaN,2067.83487,kg,5134.200,5134.20,7441.930914,20405.0,12679.075255,94356.802097,['Container'],['Container'],367.127861,366.315376,366.399675,366.399675,366.775024,366.775024,385.049116,696.25209,416.416426,395.448734,396.695752,405.009155,376.569768,675.928500,443.097388,409.140784,398.835166,418.305378,369.133404,750.886622,553.679721,485.215925,447.418634,436.652047,361.832809,793.660196,650.712980,599.852370,477.660771,468.851325,354.669782,815.397850,789.364879,803.127549,509.343868,492.873141,347.363522,797.988693,748.178216,810.241774,630.898650,502.253350,340.482034,699.643985,738.510237,770.199911,551.608249,508.225747,333.465215,643.142722,650.191255,749.140510,631.656206,511.986339,3735
1,1,430,Liberia,324,Guinea,Import,Sea,690721,690721 | Ceramic tiles; flags,6907,6907 | Ceramic flags and pa,69,69 | Ceramic products,NaN,NaN,NaN,NaN,NaN,5746.21145,NaN,5746.21145,mÃÂ²,1.902,16458.64,13112.854529,894.0,555.505674,7284.265093,['all'],['all'],25.084711,25.046515,25.050347,25.050347,25.067678,25.067899,26.020753,42.33907,27.561459,26.156396,26.221587,27.026690,25.576889,40.997396,28.855152,27.222329,26.753885,27.705551,25.187901,45.235680,34.589798,31.662539,29.753862,28.644809,24.806065,49.274597,39.885437,37.620434,31.231215,30.324299,24.431506,49.809583,47.251208,49.031483,33.761701,31.558664,24.049460,47.803689,45.893112,48.946067,39.514719,32.019927,23.689573,43.515154,45.722064,48.502554,37.843945,32.315818,23.322640,40.949130,41.644835,47.466593,39.864007,32.508489,nan


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $378.1m (20.0%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               166161049.3  302278129.1      8.8     15.0
02 | Meat and edible meat   53482658.3   28047664.6      2.8      1.4
15 | Animal, vegetable or   32555482.4   36876379.1      1.7      1.8
22 | Beverages, spirits a   22241099.9   26217962.2      1.2      1.3
19 | Preparations of cere   21569347.5    7703533.0      1.1      0.4
03 | Fish and crustaceans   17370825.8    7791509.4      0.9      0.4
17 | Sugars and sugar con   13974064.8   27507802.6      0.7      1.4
21 | Miscellaneous edible   13064475.4    4555235.5      0.7      0.2
07 | Vegetables and certa   10887225.8   25622974.3      0.6      1.3
16 | Meat, fish, crustace    6975104.2    1572613.0      0.4      0.1
04 | Dairy produce; birds    6032040.7    3061829.9      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $350.5m (18.6%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       271064174.7  385173159.6     14.4     19.1
Food & Beverages   78990402.4   67128457.8      4.2      3.3
Fishing              436844.5     121369.9      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $328.8m (17.4%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
PCR - Processed rice           157245109.9  273134755.9      8.3     13.5
OFD - Food products n.e.c.      48255833.4   22820146.6      2.6      1.1
OMT - Meat products n.e.c.      38754502.6   19115030.2      2.1      0.9
VOL - Vegetable oils and fats   32441257.4   36837655.4      1.7      1.8
CMT - Bovine meat prods         19215162.0    9911223.8      1.0      0.5
SGR - Sugar                     12125102.4   26676458.8      0.6      1.3
V_F - Vegetables, fruit, nuts   11116240.8   25821705.1      0.6      1.3
MIL - Dairy products             5165694.2    2673478.2      0.3      0.1
PDR - Paddy rice                 1933455.6    3358412.3      0.1      0.2
OAP - Animal products n.e.c.     1004423.2     237560.1      0.1      0.0
GRO - Cereal grains n.e.c.        892437.0    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 20.0% 
GTAP 17.4% 
EORA 18.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46388/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,430,Liberia,156,China,Import,Sea,690590,690590 | Ceramic chimney-pots,6905,"6905 | Roofing tiles, chimn",69,69 | Ceramic products,3735,"Non-refractory ceramic building bricks, floori...",Mining and Quarrying,34,NMM - Mineral products n.e.c.,2067.83487,NaN,2067.83487,kg,5134.200,5134.20,7441.930914,20405.0,12679.075255,94356.802097,['Container'],['Container'],367.127861,366.315376,366.399675,366.399675,366.775024,366.775024,385.049116,696.25209,416.416426,395.448734,396.695752,405.009155,376.569768,675.928500,443.097388,409.140784,398.835166,418.305378,369.133404,750.886622,553.679721,485.215925,447.418634,436.652047,361.832809,793.660196,650.712980,599.852370,477.660771,468.851325,354.669782,815.397850,789.364879,803.127549,509.343868,492.873141,347.363522,797.988693,748.178216,810.241774,630.898650,502.253350,340.482034,699.643985,738.510237,770.199911,551.608249,508.225747,333.465215,643.142722,650.191255,749.140510,631.656206,511.986339,3735,NaN
1,1,430,Liberia,324,Guinea,Import,Sea,690721,690721 | Ceramic tiles; flags,6907,6907 | Ceramic flags and pa,69,69 | Ceramic products,NaN,NaN,NaN,NaN,NaN,5746.21145,NaN,5746.21145,mÃÂ²,1.902,16458.64,13112.854529,894.0,555.505674,7284.265093,['all'],['all'],25.084711,25.046515,25.050347,25.050347,25.067678,25.067899,26.020753,42.33907,27.561459,26.156396,26.221587,27.026690,25.576889,40.997396,28.855152,27.222329,26.753885,27.705551,25.187901,45.235680,34.589798,31.662539,29.753862,28.644809,24.806065,49.274597,39.885437,37.620434,31.231215,30.324299,24.431506,49.809583,47.251208,49.031483,33.761701,31.558664,24.049460,47.803689,45.893112,48.946067,39.514719,32.019927,23.689573,43.515154,45.722064,48.502554,37.843945,32.315818,23.322640,40.949130,41.644835,47.466593,39.864007,32.508489,nan,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $272.9m of $1887.9m (14.5%)
Volume: 400.2 of 2018.0 million kilograms (19.8%)
Value-to-Weight: 0.68 from 0.94 US$ per kg (+-27.1%)
Average distance to market (volume-weighted): 7684.6 from 4504.8 nm (+70.6%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    564  197260851.2  296841537.2     10.4     14.7
1        Meat and Prep    103   33957183.6   18071424.9      1.8      0.9
2      Sugar and Honey     93   13974064.8   27507802.6      0.7      1.4
3           Vegetables    373   11322520.7   25955138.3      0.6      1.3
4  Unprocessed Cereals     29    6982483.8   25784960.9      0.4      1.3
5                Dairy    158    5187122.5    2707554.9      0.3      0.1
6        Fats and Oils     89    3642300.6    2822447.6      0.2      0.1
7               Fruits    143     556446.2     499371.3      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,5.883028e+06,1.062591e+07,6.360914e+06,6.040434e+06,6.059443e+06,6.187127e+06,5420037.3,1.244344e+07,1.204429e+07,1.225649e+07,7.780000e+06,7.526061e+06,5096873.9,9.819055e+06,9.927522e+06,1.143755e+07,9.643282e+06,7.817240e+06
1,Fats and Oils,6.756525e+06,1.166830e+07,7.254334e+06,6.864150e+06,6.884524e+06,7.066988e+06,6276797.8,1.347563e+07,1.302130e+07,1.332727e+07,8.795130e+06,8.450488e+06,5941947.8,1.084365e+07,1.102743e+07,1.272478e+07,1.068442e+07,8.748681e+06
2,Fruits,9.299739e+05,1.660043e+06,1.003155e+06,9.526769e+05,9.555547e+05,9.766635e+05,858703.9,1.944495e+06,1.877014e+06,1.913653e+06,1.226743e+06,1.182695e+06,808959.6,1.544387e+06,1.561955e+06,1.795452e+06,1.511045e+06,1.227464e+06
3,Meat and Prep,3.003795e+07,5.431511e+07,3.248494e+07,3.084923e+07,3.094651e+07,3.159505e+07,27668036.4,6.360975e+07,6.157890e+07,6.265254e+07,3.973427e+07,3.844938e+07,26013853.4,5.017201e+07,5.072187e+07,5.844097e+07,4.927594e+07,3.994041e+07
4,Processed Cereals,1.054451e+09,1.902249e+09,1.139765e+09,1.082359e+09,1.085772e+09,1.108785e+09,971684613.5,2.228482e+09,2.156967e+09,2.195097e+09,1.394861e+09,1.348056e+09,913914491.7,1.758972e+09,1.778428e+09,2.048525e+09,1.727434e+09,1.400036e+09
5,Sugar and Honey,3.018852e+07,5.034183e+07,3.208696e+07,3.045482e+07,3.055145e+07,3.144067e+07,28216339.1,5.963905e+07,5.740325e+07,5.889398e+07,3.997660e+07,3.704064e+07,26839753.3,4.824529e+07,4.894620e+07,5.586798e+07,4.727601e+07,3.819221e+07
6,Unprocessed Cereals,4.947882e+07,8.867824e+07,5.340470e+07,5.071283e+07,5.087296e+07,5.198545e+07,45651263.3,1.039861e+08,1.006071e+08,1.024502e+08,6.546385e+07,6.303590e+07,42979649.8,8.224082e+07,8.317340e+07,9.573166e+07,8.075250e+07,6.542708e+07
7,Vegetables,5.591007e+07,1.009066e+08,6.044092e+07,5.739677e+07,5.757721e+07,5.879437e+07,51517667.0,1.181875e+08,1.143792e+08,1.164090e+08,7.393654e+07,7.149534e+07,48451750.6,9.328456e+07,9.431665e+07,1.086490e+08,9.160590e+07,7.425644e+07


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.91,0.09,0.03,0.03,0.06,1.35,1.28,1.32,0.45,0.41,0.91,0.93,1.22,0.88,0.52
1,Fats and Oils,1.35,0.14,0.03,0.04,0.09,1.98,1.85,1.94,0.69,0.60,1.35,1.40,1.86,1.30,0.77
2,Fruits,1.31,0.13,0.04,0.05,0.08,1.95,1.83,1.90,0.66,0.58,1.32,1.35,1.77,1.26,0.75
3,Meat and Prep,0.71,0.07,0.02,0.03,0.05,1.06,1.00,1.03,0.36,0.32,0.71,0.73,0.95,0.69,0.41
4,Processed Cereals,4.30,0.43,0.14,0.16,0.28,6.37,6.01,6.20,2.15,1.91,4.28,4.38,5.75,4.12,2.46
5,Sugar and Honey,1.44,0.14,0.02,0.03,0.09,2.25,2.09,2.20,0.84,0.63,1.53,1.58,2.08,1.46,0.81
6,Unprocessed Cereals,5.61,0.56,0.18,0.20,0.36,8.35,7.87,8.13,2.84,2.49,5.62,5.76,7.55,5.41,3.21
7,Vegetables,3.97,0.40,0.13,0.15,0.25,5.89,5.55,5.73,1.98,1.76,3.96,4.05,5.32,3.81,2.28
